# 🔧 Notebook 04 — Data Preprocessing

---

## Project Context

| Field | Detail |
|---|---|
| **Competition** | SATRIA DATA 2026 — Big Data Challenge |
| **Experiment** | Baseline CNN — Experiment 01 |
| **Stage** | Preprocessing (Step 2 of 7) |

---

## Tujuan Tahap Preprocessing

Tahap ini bertugas mengubah data mentah menjadi format yang siap dikonsumsi model:
1. Scanning gambar corrupt (dari EDA)
2. Melakukan Stratified Split (80% Train, 20% Val)
3. Menetapkan Transformasi (Resize, Crop, Normalize, ColorJitter)
4. Menggunakan WeightedRandomSampler (untuk mengatasi imbalanced dataset)
5. Menghasilkan DataLoader yang efisien

> **Catatan:** Sesuai `CLAUDE.md`, notebook ini hanya berfungsi sebagai antarmuka (orchestrator). Seluruh implementasi teknis berada di modul `src/preprocessing/`.

In [ ]:
# ============================================================
# CELL 1 — IMPORTS & SETUP
# ============================================================
import sys
import logging
from pathlib import Path
import yaml

# Tambahkan root proyek ke sys.path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.seed import set_seed
from src.preprocessing.splitter import collect_dataset, stratified_split, save_split_report, get_split_summary
from src.preprocessing.dataloader import build_dataloaders, verify_dataloader

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger(__name__)

with open(PROJECT_ROOT / 'configs' / 'baseline.yaml', 'r') as f:
    config = yaml.safe_load(f)

SEED = config['experiment']['seed']
set_seed(SEED)
logger.info(f"Seed set to {SEED}")

---
## Step 1 — Collect Data & Stratified Split
Kami membaca seluruh direktori data dan membaginya 80/20 secara *stratified in-memory* (tanpa menyalin file ke disk).

In [ ]:
# ============================================================
# CELL 2 — DATA SPLITTING
# ============================================================
data_root = PROJECT_ROOT / config['data']['raw_dir']
train_dir = data_root / config['data']['train_subdir']

df_all = collect_dataset(train_dir)
logger.info(f"Total data terkumpul: {len(df_all)}")

df_train, df_val = stratified_split(
    df_all, 
    train_ratio=config['split']['train_ratio'], 
    val_ratio=config['split']['val_ratio'], 
    seed=SEED
)

reports_dir = PROJECT_ROOT / config['output']['reports_dir']
save_split_report(df_train, df_val, reports_dir / 'split_report.csv')

print("\nRingkasan Split:")
print(get_split_summary(df_train, df_val))

---
## Step 2 — Build DataLoaders
DataLoader mengonversi DataFrame ke tensor iterator. `train_loader` otomatis dilengkapi dengan `WeightedRandomSampler` untuk mengatasi ketidakseimbangan kelas.

In [ ]:
# ============================================================
# CELL 3 — DATALOADER INITIALIZATION
# ============================================================
train_loader, val_loader = build_dataloaders(
    df_train=df_train,
    df_val=df_val,
    config=config,
    use_weighted_sampler=True
)

print("\nVerifikasi Train Loader:")
verify_dataloader(train_loader, split_name='train', n_batches=1)

print("\nVerifikasi Val Loader:")
verify_dataloader(val_loader, split_name='val', n_batches=1)